In [41]:
import pandas as pd

df = pd.read_parquet('../data/raw/pitching_stats.parquet')

In [31]:
# Columns of interest:
# 'Stuff' -> whats the speed and movement look like
# Location - Loc+
# Velocity = v[pitch_type] for pitch type below
# Horizontal movement = [pitch_type]-X
# Vertical movement = [pitch_type]-Z
#    Four Seam (FA), Two Seam (FT) Cutters (FC), Split Fingers (FS), Forkballs (FO)
#    Sinkers (SI), Sliders (SL), Curveballs (CU), Knuckle-Curves (KC), Changeups (CH), Screwballs (SC), Knuckleballs (KN)
#  
# Location  - TODO -> integrate Trevor May's paint score!

# Batter contact effectiveness -> when batter makes contact, what's the damage
# GB%, FB%, Hard%, Barrel%, 

# Batter vision effectiviness -> how well is the batter seeing the ball against this pitcher? Are they getting fooled?
# K%, BB%, HR/FB, SwStr%, O-Contact%, O-Swing%, Contact%

In [42]:
# Feature selection
# Select base feature columns
pitches=['FA', 'FT', 'FC', 'FS', 'FO', 'SI', 'SL', 'CU', 'KC', 'CH', 'SC', 'KN']
fc_velo=[f'v{p} (sc)' for p in pitches]
fc_hmovement=[f'{p}-X (sc)' for p in pitches]
fc_vmovement=[f'{p}-Z (sc)' for p in pitches]
fc_batter_contact=['ERA','GB%', 'Hard%', 'Barrel%',]
fc_batter_vision=['K%', 'BB%','HR/FB', 'SwStr%', 'O-Swing% (sc)', 'O-Contact% (sc)', 'Contact% (sc)']
fc_x_factor=['Age', 'FIP']



feature_columns = fc_velo + fc_hmovement + fc_vmovement +  fc_batter_contact + fc_batter_vision + fc_x_factor

use_plus_stats = False
if use_plus_stats:
    for col in feature_columns.copy():
        if f'{col}+' in df.columns:
            feature_columns.append(f'{col}+')
            feature_columns.remove(col)


In [43]:
# Feature engineering
ps = df.copy()

ps = ps.sort_values(['IDfg', 'Season'])

lag_3yr_cols = fc_batter_contact + fc_velo + fc_hmovement + fc_vmovement
new_cols = {}

for col in lag_3yr_cols:
    t1 = ps.groupby('IDfg')[col].shift(1)
    t2 = ps.groupby('IDfg')[col].shift(2)
    new_cols[f'{col}_t1'] = t1
    new_cols[f'{col}_t2'] = t2
    new_cols[f'{col}_delta_1yr'] = ps[col] - t1
    feature_columns += [f'{col}_delta_1yr', f'{col}_t1', f'{col}_t2']

ps = pd.concat([ps, pd.DataFrame(new_cols, index=ps.index)], axis=1)
ps['ERA_next'] = ps.groupby('IDfg')['ERA'].shift(-1)

In [44]:
# Data cleanup
# Only entries with more than 15 innings pitched
ps = ps[ps['IP']>10]
# Only pitchers that were primarily starters (started more than half their games)
#ps['start_pct'] = ps['GS'] / ps['G']
#ps = ps[ps['start_pct'] > 0.5]

In [45]:
import json
# Save the processed and filtered stats with all of their columns so we can use the columns later if needed
ps.to_parquet('../data/raw/processed_pitch_stats.parquet')
with open('../data/raw/feature_columns.json', 'w') as file:
    json.dump(feature_columns, file, indent=4)